# Fastmagic Colab Benchmark Notebook

# AI-assisted: Claude, 2026-04-23
This notebook is designed for rigorous IQL replication: preset hyperparameters, multi-seed sweeps, CSV aggregation, and backup of checkpoints/results to Google Drive.e.

In [ ]:
# Build isolated Python 3.10 environment for D4RL (<3.11 requirement)
!apt-get update -qq
!apt-get install -y -qq libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev patchelf wget

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -f -p /usr/local
!rm -f /tmp/miniconda.sh

!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!/usr/local/bin/conda create -y -n fastmagic python=3.10

ENV_PY = '/usr/local/envs/fastmagic/bin/python'
ENV_PIP = '/usr/local/envs/fastmagic/bin/pip'

!$ENV_PIP install -q --upgrade pip setuptools wheel
!$ENV_PIP install -q torch==2.3.1 numpy==1.26.4 wandb==0.17.6 tqdm==4.66.5 pandas==2.2.2 "Cython<3"
!$ENV_PIP install -q gym==0.23.1 mujoco-py==2.1.2.14
!$ENV_PIP install -q git+https://github.com/aravindr93/mjrl.git
!$ENV_PIP install -q git+https://github.com/rail-berkeley/d4rl@master

# MuJoCo 2.1 binaries required by mujoco-py
!mkdir -p /root/.mujoco
!wget -q https://mujoco.org/download/mujoco210-linux-x86_64.tar.gz -O /tmp/mujoco210.tar.gz
!tar -zxf /tmp/mujoco210.tar.gz -C /root/.mujoco
!rm -f /tmp/mujoco210.tar.gz

import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['D4RL_SUPPRESS_IMPORT_ERROR'] = '1'
os.environ['MUJOCO_PY_MUJOCO_PATH'] = '/root/.mujoco/mujoco210'
os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin'

print('Using Python:', ENV_PY)
!$ENV_PY --version

In [ ]:
# Validate gym + D4RL using the Python 3.10 environment
import os
import subprocess

validation_code = """
import gym
import d4rl
print('gym', gym.__version__)
env = gym.make('halfcheetah-medium-v2')
dataset = d4rl.qlearning_dataset(env)
print('obs shape', dataset['observations'].shape)
env.close()
print('D4RL Mujoco dataset check: OK')
"""

validation_env = {
    **os.environ,
    'MUJOCO_GL': 'egl',
    'D4RL_SUPPRESS_IMPORT_ERROR': '1',
    'MUJOCO_PY_MUJOCO_PATH': '/root/.mujoco/mujoco210',
    'LD_LIBRARY_PATH': os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin',
}

result = subprocess.run(
    [ENV_PY, '-c', validation_code],
    capture_output=True,
    text=True,
    env=validation_env,
    check=False,
    cwd='/content',
)

print(result.stdout)
if result.returncode != 0:
    print('--- stderr ---')
    print(result.stderr)
    raise RuntimeError('Validation failed in Python 3.10 env (see stderr above)')

In [ ]:
# Optional GPU sanity check
!nvidia-smi

Fri Apr 24 01:11:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lis

In [ ]:
%cd /content
import shutil
from pathlib import Path

REPO_ROOT = Path('/content/fastmagic')

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

!git clone https://github.com/OwenLi729/fastmagic {REPO_ROOT}
%cd /content/fastmagic
print('Repo root:', REPO_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Benchmark configuration
from pathlib import Path

PRESET = 'mujoco'  # 'mujoco' or 'antmaze'
SEEDS = [0, 1, 2]
MAX_ENVS = 1  # set to None for full sweep
TRAIN_STEPS = 100000  # increase to 1000000 for paper-length runs
PROFILE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/fastmagic_benchmarks')

PRESET_ENVS = {
    'mujoco': [
        'halfcheetah-medium-v2', 'halfcheetah-medium-replay-v2', 'halfcheetah-medium-expert-v2',
        'hopper-medium-v2', 'hopper-medium-replay-v2', 'hopper-medium-expert-v2',
        'walker2d-medium-v2', 'walker2d-medium-replay-v2', 'walker2d-medium-expert-v2',
    ],
    'antmaze': [
        'antmaze-umaze-v2', 'antmaze-umaze-diverse-v2', 'antmaze-medium-play-v2',
        'antmaze-medium-diverse-v2', 'antmaze-large-play-v2', 'antmaze-large-diverse-v2',
    ],
}

SELECTED_ENVS = PRESET_ENVS[PRESET][:MAX_ENVS] if MAX_ENVS is not None else PRESET_ENVS[PRESET]
print({'preset': PRESET, 'envs': SELECTED_ENVS, 'seeds': SEEDS, 'train_steps': TRAIN_STEPS})

/content
Cloning into 'fastmagic'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 17.97 KiB | 2.57 MiB/s, done.
/content/fastmagic
/bin/bash: /usr/local/bin/pip: /usr/local/bin/python3.10: bad interpreter: No such file or directory


In [ ]:
# Cache selected datasets with Python 3.10 env
import os
import subprocess

repo_root = str(REPO_ROOT) if 'REPO_ROOT' in globals() else '/content/fastmagic'

for env_name in SELECTED_ENVS:
    print(f'Caching dataset: {env_name}')
    result = subprocess.run(
        [ENV_PY, 'data/download_d4rl.py', '--env', env_name],
        capture_output=True,
        text=True,
        cwd=repo_root,
        env={
            **os.environ,
            'MUJOCO_GL': 'egl',
            'D4RL_SUPPRESS_IMPORT_ERROR': '1',
            'MUJOCO_PY_MUJOCO_PATH': '/root/.mujoco/mujoco210',
            'LD_LIBRARY_PATH': os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin',
        },
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f'Failed to download {env_name}')

print('All selected datasets cached successfully.')

{'preset': 'mujoco', 'envs': ['halfcheetah-medium-v2', 'halfcheetah-medium-replay-v2', 'halfcheetah-medium-expert-v2', 'hopper-medium-v2', 'hopper-medium-replay-v2', 'hopper-medium-expert-v2', 'walker2d-medium-v2', 'walker2d-medium-replay-v2', 'walker2d-medium-expert-v2'], 'seeds': [0, 1, 2], 'mixed_precision': True}
{'preset': 'mujoco', 'envs': ['halfcheetah-medium-v2', 'halfcheetah-medium-replay-v2', 'halfcheetah-medium-expert-v2', 'hopper-medium-v2', 'hopper-medium-replay-v2', 'hopper-medium-expert-v2', 'walker2d-medium-v2', 'walker2d-medium-replay-v2', 'walker2d-medium-expert-v2'], 'seeds': [0, 1, 2], 'mixed_precision': True}


In [ ]:
# Run baseline first (standard IQL, no optimizations)
import os
import subprocess
import sys
from pathlib import Path

baseline_command = [
    ENV_PY, 'src/benchmark_iql.py',
    '--preset', PRESET,
    '--seeds', *[str(seed) for seed in SEEDS],
    '--train_steps', str(TRAIN_STEPS),
    '--results_root', 'results/benchmarks_baseline',
    '--checkpoint_root', 'models/benchmarks_baseline',
    '--baseline',
    '--replay_device', 'cpu',
]
if MAX_ENVS is not None:
    baseline_command.extend(['--max_envs', str(MAX_ENVS)])
if PROFILE:
    baseline_command.append('--profile')

print('Running baseline command:', baseline_command)

log_path = Path('/content/fastmagic/results/benchmarks_baseline')
log_path.mkdir(parents=True, exist_ok=True)
log_file = log_path / 'run.log'

bench_env = {
    **os.environ,
    'MUJOCO_GL': 'egl',
    'D4RL_SUPPRESS_IMPORT_ERROR': '1',
    'MUJOCO_PY_MUJOCO_PATH': '/root/.mujoco/mujoco210',
    'LD_LIBRARY_PATH': os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin',
}

with log_file.open('w') as log_fh:
    proc = subprocess.Popen(
        baseline_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=bench_env,
        cwd='/content/fastmagic',
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        log_fh.write(line)
    proc.wait()

if proc.returncode != 0:
    print(f'\n--- BASELINE FAILED (exit {proc.returncode}) — full log: {log_file} ---')
    raise RuntimeError(f'Baseline benchmark failed with exit code {proc.returncode}')
print('Baseline benchmark complete.')

Caching dataset: halfcheetah-medium-v2


CalledProcessError: Command '['python', 'data/download_d4rl.py', '--env', 'halfcheetah-medium-v2']' returned non-zero exit status 1.

In [ ]:
# Run improved config (mixed precision + GPU replay)
import os
import subprocess
import sys
from pathlib import Path

improved_command = [
    ENV_PY, 'src/benchmark_iql.py',
    '--preset', PRESET,
    '--seeds', *[str(seed) for seed in SEEDS],
    '--train_steps', str(TRAIN_STEPS),
    '--results_root', 'results/benchmarks_improved',
    '--checkpoint_root', 'models/benchmarks_improved',
    '--mixed_precision',
    '--replay_device', 'gpu',
]
if MAX_ENVS is not None:
    improved_command.extend(['--max_envs', str(MAX_ENVS)])
if PROFILE:
    improved_command.append('--profile')

print('Running improved command:', improved_command)

log_path = Path('/content/fastmagic/results/benchmarks_improved')
log_path.mkdir(parents=True, exist_ok=True)
log_file = log_path / 'run.log'

bench_env = {
    **os.environ,
    'MUJOCO_GL': 'egl',
    'D4RL_SUPPRESS_IMPORT_ERROR': '1',
    'MUJOCO_PY_MUJOCO_PATH': '/root/.mujoco/mujoco210',
    'LD_LIBRARY_PATH': os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin',
}

with log_file.open('w') as log_fh:
    proc = subprocess.Popen(
        improved_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=bench_env,
        cwd='/content/fastmagic',
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        log_fh.write(line)
    proc.wait()

if proc.returncode != 0:
    print(f'\n--- IMPROVED FAILED (exit {proc.returncode}) — full log: {log_file} ---')
    raise RuntimeError(f'Improved benchmark failed with exit code {proc.returncode}')
print('Improved benchmark complete.')

In [ ]:
# Compare baseline vs improved and copy artifacts to Drive
import pandas as pd
import shutil
from pathlib import Path

baseline_dir = Path('results/benchmarks_baseline')
improved_dir = Path('results/benchmarks_improved')

baseline_agg_path = baseline_dir / f'{PRESET}_aggregate.csv'
improved_agg_path = improved_dir / f'{PRESET}_aggregate.csv'

baseline_df = pd.read_csv(baseline_agg_path).rename(columns={
    'final_score_mean': 'baseline_final_mean',
    'best_score_mean': 'baseline_best_mean',
    'final_score_std': 'baseline_final_std',
    'best_score_std': 'baseline_best_std',
})
improved_df = pd.read_csv(improved_agg_path).rename(columns={
    'final_score_mean': 'improved_final_mean',
    'best_score_mean': 'improved_best_mean',
    'final_score_std': 'improved_final_std',
    'best_score_std': 'improved_best_std',
})

comparison_df = baseline_df.merge(improved_df, on='env', how='inner')
comparison_df['delta_final_mean'] = comparison_df['improved_final_mean'] - comparison_df['baseline_final_mean']
comparison_df['delta_best_mean'] = comparison_df['improved_best_mean'] - comparison_df['baseline_best_mean']
display(comparison_df.sort_values('env'))

drive_target = DRIVE_ROOT / PRESET
drive_target.mkdir(parents=True, exist_ok=True)

for source in [Path('results/benchmarks_baseline'), Path('results/benchmarks_improved'), Path('models/benchmarks_baseline'), Path('models/benchmarks_improved')]:
    if source.exists():
        destination = drive_target / source.name
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)

comparison_df.to_csv(drive_target / f'{PRESET}_baseline_vs_improved.csv', index=False)
print(f'Copied outputs to {drive_target}')